In [1]:
#The purpose of this script is to collect labelled data for 3 classifiers:
#1) An LSTM to recognise when a strikefoot is happening
#2) A Random Forest classifier to determine the strikefoot type (heel, midfoot, forefoot)
#3) A classifier to determine whether overstriding is occurring

#This script combines the functionality of strikefoot_type_trainer and data_extraction_1_2_3.
#A key limitation of data_extraction_1_2_3 was that the "u" frames were not saved. This fixes that.
import cv2
import numpy as np
import json
import sys
import pandas as pd
import torch
sys.path.append("/Users/abhinavarora/Desktop/CadenceCV/src")
from numpy_encoder import NumpyEncoder
from video_extracter import extract_keypoints
from data_loader import training_df, testing_df
from torch import nn
from scipy.stats import zscore
from collections import deque
from torch.utils.data import Dataset, DataLoader

In [2]:
#Update per video
video_dir = "/Users/abhinavarora/Desktop/CadenceCV/Videos/Video12.MOV"
video_name = "Video12"

#Using extract_keypoints to obtain the essential keypoints from the video
coords = extract_keypoints(video_dir)

left_ankle_arr = coords['left_ankle']
right_ankle_arr = coords['right_ankle']
left_hip_arr = coords['left_hip']
right_hip_arr = coords['right_hip']
left_knee_arr = coords['left_knee']
right_knee_arr = coords['right_knee']
left_shoulder_arr = coords['left_shoulder']
right_shoulder_arr = coords['right_shoulder']
total_frames = coords['Frame Count']
n_detected = len(left_ankle_arr)
print(f'Keypoints detected in {n_detected}/{total_frames} frames')

#The purpose of this block is to extract the following 4 LSTM metrics across ALL frames:
#1) Velocity of the ankle x component
#2) Velocity of the ankle y component
#3) Velocity of the shin (x component of the vector between the knee and the ankle)
#4) Tibial angle
left_ankle_x_vel = np.gradient(left_ankle_arr[:, 0])
left_ankle_y_vel = np.gradient(left_ankle_arr[:, 1])
right_ankle_x_vel = np.gradient(right_ankle_arr[:, 0])
right_ankle_y_vel = np.gradient(right_ankle_arr[:, 1])

#Shin vector goes from ankle to knee
left_shin_vec = left_knee_arr - left_ankle_arr
right_shin_vec = right_knee_arr - right_ankle_arr
left_shin_velocity = np.gradient(left_shin_vec[:, 0])
right_shin_velocity = np.gradient(right_shin_vec[:, 0])

#Tibial angle: -> angle between the shin vector and the vertical (pi/2 subtracted since arctan2 is w.r.t the horizontal)
def _tibial(shin_vec_arr):
    return np.pi / 2 - np.arctan2(shin_vec_arr[:, 1], shin_vec_arr[:, 0])

left_tibial_angle = _tibial(left_shin_vec)
right_tibial_angle = _tibial(right_shin_vec)

#Draws a vertical line upward from the ankle — used to visually judge whether the foot
#lands in front of the pelvis (overstriding) or beneath it (neutral)
#extract_keypoints skips frames with no detection, so idx is clamped to the last detected frame
def draw_ankle_line(frame, actual_frame_idx, which_leg):
    idx = min(actual_frame_idx, n_detected - 1)
    ankle = left_ankle_arr[idx] if which_leg == 'left' else right_ankle_arr[idx]
    ax, ay = int(ankle[0]), int(ankle[1])
    if ax > 0:
        cv2.line(frame, (ax, ay), (ax, 0), (0, 0, 255), 2)
    return frame

#Calculates all 9 metrics at a given strikefoot frame for the given leg:
#LSTM metrics (4): ankle_x_vel, ankle_y_vel, shin_velocity, tibial_angle
#Classifier 1 metrics (5): knee flexion angle, ankle-hip horizontal offset,
#                           trunk angle, pre-strike ankle velocity, ankle approach angle
#All of these are measured at the strikefoot frame
def compute_all_metrics(actual_frame, which_leg):
    idx = min(actual_frame, n_detected - 1)
    if which_leg == 'left':
        ax_vel = left_ankle_x_vel[idx]
        ay_vel = left_ankle_y_vel[idx]
        sh_vel = left_shin_velocity[idx]
        tib = left_tibial_angle[idx]
        ankle_arr = left_ankle_arr
        knee_arr = left_knee_arr
        hip_arr = left_hip_arr
        shldr_arr = left_shoulder_arr
        all_ay = left_ankle_y_vel
    else:
        ax_vel = right_ankle_x_vel[idx]
        ay_vel = right_ankle_y_vel[idx]
        sh_vel = right_shin_velocity[idx]
        tib = right_tibial_angle[idx]
        ankle_arr = right_ankle_arr
        knee_arr = right_knee_arr
        hip_arr = right_hip_arr
        shldr_arr = right_shoulder_arr
        all_ay = right_ankle_y_vel

    #Knee flexion
    #np.arctan2 gives angles relative to the horizontal, so the angles must be subtracted
    v1 = hip_arr[idx] - knee_arr[idx]
    v2 = ankle_arr[idx] - knee_arr[idx]
    knee_flexion = float(np.rad2deg(np.arctan2(v2[1], v2[0]) - np.arctan2(v1[1], v1[0])) % 360)

    #Ankle-hip horizontal offset
    offset = float(np.abs(ankle_arr[idx, 0] - hip_arr[idx, 0]))

    #Trunk angle: -> angle between the shoulder and the hip w.r.t the vertical
    tv = shldr_arr[idx] - hip_arr[idx]
    trunk_angle = float(np.rad2deg(np.pi / 2 - np.arctan2(tv[1], tv[0])))

    #Pre-strikefoot ankle velocity (2 detected frames before the strikefoot)
    pre2 = max(0, idx - 2)
    ankle_vel_pre = float(all_ay[pre2])

    #Pre-strikefoot ankle movement direction angle
    p1 = max(0, idx - 1)
    p3 = max(0, idx - 3)
    dx = float(ankle_arr[p1, 0] - ankle_arr[p3, 0])
    dy = float(ankle_arr[p1, 1] - ankle_arr[p3, 1])
    approach_angle = float(np.arctan2(dy, dx))

    return {
        'ankle_x_vel': float(ax_vel),
        'ankle_y_vel': float(ay_vel),
        'shin_velocity': float(sh_vel),
        'tibial_angle': float(tib),
        'knee_flexion_angle': knee_flexion,
        'ankle_hip_horizontal_offset': offset,
        'trunk_angle': trunk_angle,
        'ankle_velocity_pre_strike': ankle_vel_pre,
        'ankle_approach_angle': approach_angle,
        'frame': idx,
        'side': which_leg
    }


video 1/1 (frame 1/1104) /Users/abhinavarora/Desktop/CadenceCV/Videos/Video12.MOV: 1280x736 4 persons, 1579.3ms
video 1/1 (frame 2/1104) /Users/abhinavarora/Desktop/CadenceCV/Videos/Video12.MOV: 1280x736 2 persons, 1111.6ms
video 1/1 (frame 3/1104) /Users/abhinavarora/Desktop/CadenceCV/Videos/Video12.MOV: 1280x736 2 persons, 1068.1ms
video 1/1 (frame 4/1104) /Users/abhinavarora/Desktop/CadenceCV/Videos/Video12.MOV: 1280x736 2 persons, 1053.6ms
video 1/1 (frame 5/1104) /Users/abhinavarora/Desktop/CadenceCV/Videos/Video12.MOV: 1280x736 2 persons, 1058.4ms
video 1/1 (frame 6/1104) /Users/abhinavarora/Desktop/CadenceCV/Videos/Video12.MOV: 1280x736 3 persons, 1050.0ms
video 1/1 (frame 7/1104) /Users/abhinavarora/Desktop/CadenceCV/Videos/Video12.MOV: 1280x736 2 persons, 1045.7ms
video 1/1 (frame 8/1104) /Users/abhinavarora/Desktop/CadenceCV/Videos/Video12.MOV: 1280x736 2 persons, 1045.7ms
video 1/1 (frame 9/1104) /Users/abhinavarora/Desktop/CadenceCV/Videos/Video12.MOV: 1280x736 1 person, 1

In [3]:
#This function will go through the video frame by frame. Each frame is classified as:
#"u" = the EXACT first frame where the foot begins going up (toe-off)
#"d" = the EXACT first frame where the foot contacts the ground (strikefoot)
#"s" = skip (if the frame is neither u nor d)
#"u" and "d" apply to the EXACT moment the foot goes up or down respectively.
#Each consecutive pair of "u" and "d" will be classified as a gait cycle.
def scrub_and_label(video_dir, which_leg):
    side_char = 'L' if which_leg == 'left' else 'R'
    cap = cv2.VideoCapture(video_dir)
    total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    raw_labels = []
    current_u_frame = None

    for frame_idx in range(total):
        cap.set(cv2.CAP_PROP_POS_FRAMES, frame_idx)
        ret, frame = cap.read()
        if not ret:
            break

        frame = draw_ankle_line(frame, frame_idx, which_leg)

        #Helps in identifying which foot is to be tracked
        h, w = frame.shape[:2]
        #Scale font size relative to frame width (1280 used as a reference baseline)
        scale = w / 1280
        fs = scale * 0.55
        fs_bottom = scale * 0.9
        th = max(1, int(scale * 2))
        mg = int(scale * 15)
        pink = (203, 192, 255)

        cv2.putText(frame, f'{which_leg.upper()} | Frame {frame_idx}',
                    (mg, mg + 24), cv2.FONT_HERSHEY_SIMPLEX, fs, (255, 255, 0), th, cv2.LINE_AA)
        if current_u_frame is not None:
            cv2.putText(frame, f'u_frame={current_u_frame}',
                        (mg, mg + 50), cv2.FONT_HERSHEY_SIMPLEX, fs, (200, 200, 0), th, cv2.LINE_AA)
        cv2.putText(frame, 'u=up  d=down  s=skip  c=quit',
                    (mg, h - mg), cv2.FONT_HERSHEY_SIMPLEX, fs_bottom, pink, th, cv2.LINE_AA)

        cv2.imshow('Labelling', frame)
        key = cv2.waitKey(0) & 0xFF

        #Constructing a single gait cycle — u_frame is stored until the matching d completes it
        if key == ord('u'):
            current_u_frame = frame_idx

        elif key == ord('d'):
            #"d" is always the strikefoot frame — immediately prompt for strike type then overstriding
            strike_frame = frame.copy()
            #Input classification (h = heel, m = midfoot, f = forefoot, s = skip this strike entirely)
            cv2.putText(strike_frame, 'Strike: h=heel  m=midfoot  f=forefoot  s=skip',
                        (mg, h - mg), cv2.FONT_HERSHEY_SIMPLEX, fs_bottom, pink, th, cv2.LINE_AA)
            cv2.imshow('Labelling', strike_frame)
            pk = cv2.waitKey(0) & 0xFF

            if pk == ord('s'):
                current_u_frame = None
                continue

            strike_map = {ord('h'): 'heel', ord('m'): 'midfoot', ord('f'): 'forefoot'}
            if pk not in strike_map:
                current_u_frame = None
                continue
            strike_pattern = strike_map[pk]

            #Input overstriding classification — the vertical ankle line helps judge this
            #If the line falls anterior to the pelvis it is overstriding, if within it is neutral
            over_frame = frame.copy()
            cv2.putText(over_frame, 'Overstride: o=yes  n=neutral  s=skip',
                        (mg, h - mg), cv2.FONT_HERSHEY_SIMPLEX, fs_bottom, pink, th, cv2.LINE_AA)
            cv2.imshow('Labelling', over_frame)
            ok_key = cv2.waitKey(0) & 0xFF

            overstride_map = {ord('o'): 'overstriding', ord('n'): 'neutral', ord('s'): 'skip'}
            overstriding = overstride_map.get(ok_key, 'skip')

            #Appending the completed gait cycle and resetting u_frame for the next cycle
            raw_labels.append({
                'frame': frame_idx,
                'side': side_char,
                'strike_pattern': strike_pattern,
                'overstriding': overstriding,
                'u_frame': current_u_frame,
            })
            current_u_frame = None

        elif key == ord('c'):
            break

    cap.release()
    cv2.destroyAllWindows()
    return raw_labels

#These metrics must be extracted for ALL frames in each video before scrubbing begins
#Manual labelling done via tracking the left foot first
left_raw  = scrub_and_label(video_dir, 'left')
#This time, the right foot will be focused on to identify its gait cycles
right_raw = scrub_and_label(video_dir, 'right')

In [ ]:
#Obtaining metrics for each individual strikefoot frame and combining with the raw labels
#left and right entries are merged into a single list and sorted by frame number
all_entries = []

for raw in left_raw + right_raw:
    which_leg = 'left' if raw['side'] == 'L' else 'right'
    metrics = compute_all_metrics(raw['frame'], which_leg)
    entry = {
        'frame': raw['frame'],
        'side': raw['side'],
        'strike_pattern': raw['strike_pattern'],
        'overstriding': raw['overstriding'],
        'u_frame': raw['u_frame'],
        'video': video_name,
        **metrics,
    }
    all_entries.append(entry)

all_entries.sort(key=lambda x: (x['frame'], x['side']))
print(f'Total labeled strikes: {len(all_entries)}')

In [ ]:
#Data loading loop:
#1) Read existing data from strikefoot_data.json
#2) To the same list, add the new entries for this video
#3) Finally, rewrite strikefoot_data.json with the combined data
#If the file doesn't exist or holds old-format data (a dict), start fresh as an empty list
json_path = "/Users/abhinavarora/Desktop/CadenceCV/data/strikefoot_data.json"
with open(json_path, 'r') as f:
    existing = json.load(f)

existing.extend(all_entries)
with open(json_path, 'w') as f:
    json.dump(existing, f, cls=NumpyEncoder, indent=4)

In [ ]:
video_extensions = [".mp4", ".MOV"]
base_video_dir_path = "/Users/abhinavarora/Desktop/CadenceCV/Videos"
vids = []
with open (json_path, "r") as file:
    data = json.load(file)

for frame in data:
    vids.append(frame["video"])


metadata = []
for video in vids:
    for ext in video_extensions:
        try:
            cap = cv2.VideoCapture(base_video_dir_path + "/" + video + ext)
            fps = cap.get(cv2.CAP_PROP_POS_FRAMES)
            #Gets width and heigt
            width  = cap.get(cv2.CV_CAP_PROP_FRAME_WIDTH)   
            height = cap.get(cv2.CV_CAP_PROP_FRAME_HEIGHT)
            print(width, height, fps)
            metadata.append({"Video": video, "Width": width, "Height": height, "FPS": fps})
            break
        except:
            pass

with open("/Users/abhinavarora/Desktop/CadenceCV/data/video_meta_data.json", "w") as file:
    json.dump(metadata, file, indent=4, cls=NumpyEncoder)

In [ ]:
def configure_data(strike_df, frame_by_frame_path, fit_scaler=True, scaler_means=None, scaler_stds=None):
    """
    strike_df: dataframe of labeled strikes (your training_df or test equivalent)
    frame_by_frame_path: path to frame_by_frame_data.json
    fit_scaler: True for training data, False for test data
    scaler_means, scaler_stds: pass in training stats when fit_scaler=False
    """
    lstm_metrics = ["ankle_x_vel", "tibial_angle", "shin_velocity", "ankle_y_vel"]
    footnet_metrics = lstm_metrics + ["u_frame", "video", "frame", "side"]

    #Dropping the rows where the u_frame doesn't exist (That is the only thing that doesn't exist)
    strike_df = strike_df.dropna(axis=0, how='any').copy()

    #First step: Separate the strike_df by video into different dfs
    dfs_to_merge = []
    for v in strike_df["video"].unique():
        df = strike_df[strike_df["video"] == v][footnet_metrics].copy()
        #Now, sorting items by leg, then frame. This is important to label contact and non-contact strikes effectively
        df.sort_values(by=["side", "frame"], ascending=True, inplace=True)
        dfs_to_merge.append(df)

    footnet_training_df = pd.concat(dfs_to_merge, axis=0).reset_index(drop=True)

    #Now, obtaining the data from frame_by_frame_data.json
    with open(frame_by_frame_path, "r") as file:
        data = json.load(file)

    video_metric_df = pd.concat(
        [pd.DataFrame(data[key]) for key in data], axis=0
    ).dropna(axis=0, how='any').copy()

    video_metric_df.loc[video_metric_df["side"] == "left", "side"] = "L"
    video_metric_df.loc[video_metric_df["side"] == "right", "side"] = "R"

    #Applying z_score normalisation to specific columns in video_metric_df. In accordance with FootNet's research
    #fit_scaler=True for training data, False for test data (use training stats)
    if fit_scaler:
        scaler_means = video_metric_df[lstm_metrics].mean()
        scaler_stds = video_metric_df[lstm_metrics].std()

    video_metric_df[lstm_metrics] = (
        video_metric_df[lstm_metrics] - scaler_means
    ) / scaler_stds

    #Now, adding a column to video_metric_df to indicate where there is contact and no contact between the ground
    #Non-contact points: each row in footnet_training_df's u and d frames (between those frames)
    #Contact points: done via obtaining the previous same-side row's d frame and the current row's u frame
    #Initialsing label values to None -> Indicates there isn't enough info to fill in contact or no contact
    video_metric_df["label"] = None
    gait_cycle_boundaries = []

    for i, (index, row) in enumerate(footnet_training_df.iterrows()):
        # Non-contact points
        video_metric_df.loc[
            (video_metric_df["video"] == row["video"]) &
            (video_metric_df["frame"] >= row["u_frame"]) &
            (video_metric_df["frame"] < row["frame"]) &
            (video_metric_df["side"] == row["side"]),
            "label"
        ] = 0

        # Contact points — find previous strike of same video + same side
        same_side_prev = footnet_training_df[
            (footnet_training_df["video"] == row["video"]) &
            (footnet_training_df["side"] == row["side"]) &
            (footnet_training_df["frame"] < row["frame"])
        ]

        if not same_side_prev.empty:
            prev_row = same_side_prev.iloc[-1]
            video_metric_df.loc[
                (video_metric_df["video"] == row["video"]) &
                (video_metric_df["frame"] >= prev_row["frame"]) &
                (video_metric_df["frame"] < row["u_frame"]) &
                (video_metric_df["side"] == row["side"]),
                "label"
            ] = 1

            # Capture the full cycle boundary directly here
            gait_cycle_boundaries.append({
                "video": row["video"],
                "side": row["side"],
                "start_frame": prev_row["frame"],
                "end_frame": row["frame"] - 1
            })

    # Remove cycles where end_frame < start_frame (invalid)
    valid_cycles = [c for c in gait_cycle_boundaries if c['end_frame'] >= c['start_frame']]

    Q1 = 19
    Q3 = 27
    IQR = Q3 - Q1  # 8
    upper_bound = Q3 + 1.5 * IQR  # 27 + 12 = 39
    lower_bound = max(1, Q1 - 1.5 * IQR)  # at least 1 frame

    filtered_cycles = [c for c in valid_cycles if lower_bound <= (c['end_frame'] - c['start_frame']) <= upper_bound]

    #Now, obtaining the rows for each cycle:
    video_metric_df = video_metric_df.reset_index(drop=True)
    actual_cycles = []

    for cycle in filtered_cycles:
        df = video_metric_df[
            (video_metric_df["video"] == cycle["video"]) &
            (video_metric_df["side"] == cycle["side"]) &
            (video_metric_df["frame"] >= cycle["start_frame"]) &
            (video_metric_df["frame"] <= cycle["end_frame"])
        ].copy()
        #Sorting to ensure frames are in chronological order
        df.sort_values(by=["frame"], axis=0, inplace=True)
        labels = deque(df["label"].tolist())
        actual_cycles.append((df[lstm_metrics], labels))

    #The resampling num, which is the number of timesteps the strikefoot detection LSTM will have
    resampling_num = 40

    #iterate again to resample to size resampling_num (in this case 40)
    #Mask array required to tell which ones are 0 padded: for both the labels and the gait_cycle
    all_features = []
    all_masks = []
    all_labels = []

    for gait_cycle, labels in actual_cycles:
        gait_cycle = gait_cycle.copy()
        gait_cycle_len = len(gait_cycle)
        mask_array = [True] * resampling_num
        pad_amount = resampling_num - gait_cycle_len

        for i in range(pad_amount):
            #Zeros for each feature
            mask_array[i] = False
            gait_cycle.loc[-1] = [0, 0, 0, 0]
            labels.appendleft(-1)
            gait_cycle.index += 1
            gait_cycle.sort_index(inplace=True)
            gait_cycle.reset_index(drop=True, inplace=True)

        all_features.append(gait_cycle.to_numpy())
        all_labels.append(list(labels))
        all_masks.append(mask_array)

    #Finally converting to a tensor:
    all_features = torch.from_numpy(np.array(all_features)).float()
    all_labels = torch.from_numpy(np.array(all_labels))
    all_masks = torch.from_numpy(np.array(all_masks))

    return all_features, all_labels, all_masks, scaler_means, scaler_stds

In [ ]:
# Training
train_features, train_labels, train_masks, means, stds = configure_data(
    training_df, "/Users/abhinavarora/Desktop/CadenceCV/data/frame_by_frame_data.json", fit_scaler=True
)

# Test — pass back the training stats
test_features, test_labels, test_masks, _, _ = configure_data(
    testing_df, "/Users/abhinavarora/Desktop/CadenceCV/data/frame_by_frame_data.json", fit_scaler=False,
    scaler_means=means, scaler_stds=stds
)

In [ ]:
class CustomDataLoader(Dataset):
    def __init__(self, features, labels, masks):
        super(CustomDataLoader, self).__init__()
        #Loading the data from the tensor
        self.features:torch.Tensor = features
        self.labels:torch.Tensor = labels
        self.masks:torch.Tensor = masks
    def __getitem__(self, index):
        return self.features[index].float(), self.labels[index].float(), self.masks[index].bool()

    def __len__(self):
        return self.features.shape[0]

dataset = CustomDataLoader(train_features, train_labels, train_masks)
test_dataset = CustomDataLoader(test_features, test_labels, test_masks)
first_features, first_labels, first_masks = dataset[0]

#batch_side loads in 4 batches at a time, shuffle allows to randomly shuffle the batches
train_data_loader = DataLoader(dataset=dataset, batch_size=32, shuffle=True)
test_data_loader = DataLoader(dataset=test_dataset, batch_size=32, shuffle=True)

In [ ]:
#Finally constructing the LSTM to detect strikefoot 
#Input would be a tensor with shape: 
#(batch size (an X number of gait cycles will be fed at a time), A sequence length (the no. of hidden states), Input size = number of features there exist)
#Hidden size is the length of each hidden state vector

class LSTM_custom(nn.Module):
    def __init__(self, input_size, hidden_size, num_layers, num_classes):
        super(LSTM_custom, self).__init__()
        self.input_size = input_size
        self.hidden_size = hidden_size
        self.num_layers = num_layers
        self.num_classes = num_classes
        self.lstm = nn.LSTM(input_size=self.input_size, hidden_size=self.hidden_size, num_layers=self.num_layers, batch_first=True, bidirectional=True)
        #Input will have the following shape (32, 40, 4) for 32 gait cycles, 40 timesteps, 4 input features
        #nn.Linear outputs a linear transformation with the inpout tensor as (..., hidden_size) and output tensor of (..., num_classes)
        #self.fc will be applied at the time of prediction since the output should have the final dimension as the number of classes
        #Hidden_size * 2 is required since bidrectional=True doubles the input_size
        self.fc = nn.Linear(hidden_size * 2, num_classes)

    def forward(self, x):
        #Hidden state and cell state (initial values are managed by default)
        out, _ = self.lstm(x)
        #Applying a linear transformation 
        #Out shape: (batch_size, sequence_length, hidden_size)
        output = self.fc(out)
        #output shape: (batch_size, 40, num_classes)
        return output.squeeze(-1)
    

model = LSTM_custom(input_size=4, hidden_size=32, num_layers=1, num_classes=1)

# Hyperparameters
num_epochs = 100
learning_rate = 1e-3

# Class imbalance: compute pos_weight from training data
num_negative = (train_labels[train_masks] == 0).sum().float()
num_positive = (train_labels[train_masks] == 1).sum().float()
#For balancing out classes 
pos_weight = torch.tensor([num_negative / num_positive])

#Binary cross entropy loss. For softmax regression.
criterion = nn.BCEWithLogitsLoss(reduction='none', pos_weight=pos_weight)
#Adam_optimizer
optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)

print(f"Num negative: {num_negative}")
print(f"Num positive: {num_positive}")
print(f"pos_weight: {pos_weight}")

for epoch in range(num_epochs):
    #Puts the model in training mode. Essential for applying dropout or batch normalisation. They are different during training and testing
    model.train()
    #Resetting loss after each epoch to see if it decreases
    total_loss = 0

    for features, labels, masks in train_data_loader:
        #Clears gradients from the previous input batch
        optimizer.zero_grad()
        
        #Cloning the labels so the -1s can be zeroed out
        safe_labels = labels.clone()
        safe_labels[~masks] = 0

        #Inputting the features into the LSTM model -> this is the forward pass
        logits = model(features)
        #Calculating the loss
        loss = criterion(logits, safe_labels.float())
        #Calculating loss only over the mas
        loss = (loss * masks.float()).sum() / masks.sum()

        #Backprop function
        loss.backward()
        #Updateing the weihts
        optimizer.step()
        #Summing up the loss over the batches
        total_loss += loss.item()


In [ ]:
model.eval()
all_preds = []
all_true = []

with torch.no_grad():
    for features, labels, masks in test_data_loader:
        logits = model(features)
        preds = (torch.sigmoid(logits) > 0.35).long()

        # Only evaluate on real frames, not padding
        all_preds.append(preds[masks])
        all_true.append(labels[masks])

all_preds = torch.cat(all_preds).numpy()
all_true = torch.cat(all_true).numpy()

from sklearn.metrics import classification_report
print(classification_report(all_true, all_preds, target_names=["non-contact", "contact"]))



In [ ]:
torch.save(model.state_dict(), "/Users/abhinavarora/Desktop/CadenceCV/models/footnet_lstm_best.pth")